## Step1:Importing Modules

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *
import time

Insight: Importing the required modules

## Step2:Create Spark Session

In [0]:
sess=SparkSession.builder.appName("celebal_assignment").getOrCreate()


Insight: Create spark session using SparkSession 

### Manual Schema For Dataset

In [0]:
schm = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("order_date", StringType(), True),            # Manual Schema 
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", FloatType(), True),
    StructField("payment_mode", StringType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("rating",FloatType(), True),
    StructField("returned", BooleanType(), True)
])

Insight:Creating the manual schema for the dataset which will  loaded into Spark Dataframe

## Step3:Load Dataset

In [0]:
data=spark.read.format("csv")\
    .option("header","true")\
        .schema(schm)\
            .load("/Volumes/dbacademy/default/data/retail_orders.csv")

Insight:Loading the dataset into Spark Dataframe

## Step4:Exploration of Dataset

Insight:Exploring the basic information of the dataset.

In [0]:
shape=data.count(),len(data.columns)
print(f"No of rows: {shape[0]}")
print(f"No of columns:{shape[1]}")




No of rows: 1100
No of columns:15


In [0]:
print("Columns of dataset:")
print()
data.columns

Columns of dataset:



['order_id',
 'customer_fullname',
 'gender',
 'age',
 'city',
 'state',
 'order_date',
 'product',
 'category',
 'quantity',
 'unit_price',
 'payment_mode',
 'delivery_status',
 'rating',
 'Total_amount',
 'is_delivered']

In [0]:
display(data.limit(5))

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned
1,Connie Boyd,Male,41,Delhi,Delhi,24/11/2023,Phone,Electronics,2,599.0,null,Delivered,-1.0,false
2,Daniel Jimenez,Male,30,Delhi,Karnataka,01-17-2023,Laptop,Electronics,-2,599.0,Crypto,Pending,-1.0,false
3,Mark Young,F,41,Jaipur,Delhi,2023-04-23,Headphones,Accessories,0,29999.0,Cash,Pending,4.0,false
4,Richard Smith,M,22,Mumbai,Maharashtra,22/02/2023,Phone,Electronics,-2,29999.0,UPI,null,-1.0,true
5,Marcus Wang,Male,150,Mumbai,Karnataka,05-13-2024,Printer,Electronics,3,1499.0,null,Delivered,3.0,null


In [0]:
data.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: float (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- returned: boolean (nullable = true)



In [0]:
# %md
# Insight:
# Checking the shape of the dataset
# Showing all the columns of the dataset
# Displaying the few records of the dataset
# Verified the schema of the dataset




## Step5:Data Cleaning

### 5.1:Selecting useful columns

In [0]:
data=data.select([col for col in data.columns if col !="returned"])


Insight:Selecting the required columns for further processing.

### 5.2:Casting column's Datatype

In [0]:
data=data.withColumn("rating",col("rating").cast("int"))\
    .withColumn("unit_price",col("unit_price").cast("int"))\
    .withColumn("quantity",col("quantity").cast("int"))\
        .withColumn("age",col("age").cast("int"))


Insight:Converted selected columns to appropriate  data types for calculations.

In [0]:
data.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- rating: integer (nullable = true)



### 5.3:Checking Null Values

In [0]:
null_values = data.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in data.columns
])

display(null_values)

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating
0,98,0,79,36,42,0,30,27,0,0,368,288,59


Insight:Counted missing values in each column to identify data quality.

## 5.4:Handling Null Values

### 5.4.1:Droping the nan values

In [0]:
data=data.dropna(subset="customer_name")  

Insight: Removing record where the customer name was missing

### 5.4.2:Filling the nan values

In [0]:
age_mean=data.select(mean("age")).collect()[0][0]
rating_mean=data.select(mean("rating")).collect()[0][0]
data=data.fillna({
    "age":age_mean,
    "rating":rating_mean,
    "unit_price":0,
    "quantity":0,
    "city":"unknown",
    "state":"unknown",
    "product":"unknown",
    "category":"unknown",
    "payment_mode":"unknown",
    "delivery_status":"unknown"
})

Insight:Filling null values with suitable values.

### 5.5:Handling the column's values

In [0]:
data=data.withColumn("unit_price", when(col("unit_price")<0,-1*col("unit_price"))\
    .otherwise(col("unit_price")))
data=data.withColumn("quantity",when(col("quantity")<0,lit(0))\
    .otherwise(col("quantity")))

data=data.withColumn("rating",when(col("rating")<0,lit(0))\
    .otherwise(col("rating")))



Insight:Handling the values of each column with respect to correct data type.

### 5.6:Checking null values

In [0]:
null_values = data.select([
    count(when(col(x).isNull(), x)).alias(x)
    for x in data.columns
])

display(null_values)

order_id,customer_name,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating
0,0,0,0,0,0,0,0,0,0,0,0,0,0


Insight:Again checking null values after handling the null values.

### 5.7:Drop Duplicates

Insight:Removing the duplicate record to imporove data quality.

#### 5.7.1:Count Before Removing Duplicates From Dataset

In [0]:
data.count()

1002

In [0]:
data=data.drop_duplicates()

#### 5.7.2:Count After Removing Duplicates From Dataset

In [0]:
data.count()

910

### 5.8:Renaming The Columns

In [0]:
data=data.withColumnRenamed("customer_name","customer_fullname")

Insight: Renaming the customer name as customer_fullname

### 5.9:Sort The Dataset

In [0]:
data=data.sort(col("order_id"))

Insight: Sorted the dataset by order id 

## Step6:Transformations

### 6.1:Narrow Transformation

#### 6.1.1:WithColumn (Adding New Column)

In [0]:

data=data.withColumn("Total_amount",col("quantity")*col("unit_price"))
data=data.withColumn("is_delivered",when(col("delivery_status")=="Delivered",1)\
    .otherwise(0))



Insight:Created new columns such as total amount and delivery status

#### 6.1.2:Filter And Select

In [0]:
df=data.filter(col("is_delivered")==1)\
    .select("order_id","customer_fullname","Total_amount")
display(df)

order_id,customer_fullname,Total_amount
1,Connie Boyd,1198
5,Marcus Wang,4497
8,Nathan Marshall,1500
10,Tiffany James DDS,1000
14,Jamie Drake,1500
18,Michael Jenkins,1499
20,Derrick Carson,0
25,Edward Moore,0
26,Gary Ramsey,1198
31,Sarah Chavez,0


Insight:Showing only those products which are delivered. 

In [0]:
df=data.filter((col("state")=="Rajasthan")&(col("is_delivered")==1)&(col("city")=="Jaipur"))\
    .select("order_id","customer_fullname","Total_amount")
display(df)
print(f"Customers ordered from jaipur: {df.count()}")

order_id,customer_fullname,Total_amount
18,Michael Jenkins,1499
103,Vincent Stokes PhD,1500
246,Michael West,59998
395,Cody Charles,59998
481,Aaron Mayer,1797
539,David Murphy,1797
573,Brian Savage,599
626,Denise Hendrix,0
727,Lynn Weiss,0
787,Michael Reed,1797


Customers ordered from jaipur: 12


Insight:Showing those customers who are from jaipur in rajasthan and whose products are delivered with their total amount.

In [0]:
df=data.filter(col("quantity")>2)\
    .select("order_id","customer_fullname","state","city","Total_amount")
display(df)

order_id,customer_fullname,state,city,Total_amount
5,Marcus Wang,Karnataka,Mumbai,4497
6,Brianna Dennis,Gujarat,Jaipur,1500
8,Nathan Marshall,Rajasthan,Delhi,1500
12,Debbie Bailey,Rajasthan,Mumbai,1500
14,Jamie Drake,Karnataka,Jaipur,1500
30,Mrs. Brittany Rodriguez,Delhi,Jaipur,1797
32,Jessica Evans,Delhi,Mumbai,4497
37,Kelly Evans,Rajasthan,Mumbai,89997
38,John Holt,Karnataka,Delhi,1797
40,Jesse Larson,Delhi,Bengaluru,1500


Insight:Showing the customers details with total amount who ordered quantity of products more than 2.

In [0]:
df=data.filter((col("rating")>4)&(col("quantity")>=2)&(col("is_delivered")==1))\
    .select("order_id","customer_fullname","quantity","category")
display(df)

order_id,customer_fullname,quantity,category
14,Jamie Drake,3,Accessories
26,Gary Ramsey,2,Accessories
42,Angela Anderson,3,Electronics
80,Judy Gonzalez,3,Electronics
146,Joshua Sanchez,3,Electronics
243,Daniel Ford,2,Electronics
283,Brian Reyes,2,Electronics
292,Ashley Rodriguez,2,Accessories
299,Kimberly Allen,3,Electronics
316,Claudia Smith,3,Accessories


Insight:Showing those customers who give product rating more 4 and quantity ordered more than 2 and are delivered.

## 6.2:Wide Transformations

### 6.2.1:Group By And OrderBy

In [0]:
df=data.filter(col("is_delivered")==1)
df.groupBy("city").agg(sum("Total_amount").alias("Total_amount"),sum("quantity").alias("Total_quantity"),count("order_id").alias("Total_orders")).orderBy(col("Total_amount").desc()).show()

+---------+------------+--------------+------------+
|     city|Total_amount|Total_quantity|Total_orders|
+---------+------------+--------------+------------+
|Ahmedabad|      453157|            53|          46|
|   Mumbai|      434948|            61|          50|
|Bengaluru|      411770|            46|          52|
|    Delhi|      406850|            64|          48|
|   Jaipur|      369356|            58|          39|
|  unknown|       61998|             6|           7|
+---------+------------+--------------+------------+



Insight:Calculated total sales , quantity and total orders which are delivered city wise.

In [0]:
df=data.filter(col("is_delivered")==0)
df.groupBy("city").agg(sum("Total_amount").alias("Total_amount"),sum("quantity").alias("Total_quantity"),count("order_id").alias("Total_orders")).orderBy(col("Total_amount").desc()).show()

+---------+------------+--------------+------------+
|     city|Total_amount|Total_quantity|Total_orders|
+---------+------------+--------------+------------+
|   Mumbai|     1868563|           185|         141|
|    Delhi|     1383486|           135|         105|
|Bengaluru|     1233071|           175|         147|
|Ahmedabad|     1090095|           147|         135|
|   Jaipur|      626118|           120|         117|
|  unknown|      291478|            35|          23|
+---------+------------+--------------+------------+



Insight:Calculated total sales , quantity and total orders which are not delivered city wise.

In [0]:
df=data.filter(col("is_delivered")==1)
df.groupBy("category").agg(sum("Total_amount").alias("Total_amount"),sum("quantity").alias("Total_quantity"),count("order_id").alias("Total_orders")).orderBy(col("Total_amount").desc()).show()

+-----------+------------+--------------+------------+
|   category|Total_amount|Total_quantity|Total_orders|
+-----------+------------+--------------+------------+
|Accessories|     1204008|           114|          92|
|Electronics|      861783|           162|         141|
|    unknown|       72288|            12|           9|
+-----------+------------+--------------+------------+



Insight:Calculated total sales , quantity and total orders which are delivered category wise.

#### 6.2.2:Distinct

In [0]:
data.select("city").distinct().show()

+---------+
|     city|
+---------+
|    Delhi|
|   Jaipur|
|Ahmedabad|
|   Mumbai|
|Bengaluru|
|  unknown|
+---------+



Insight:Displayed the unique cities available in the dataset.

## Step7:Csv File VS Parquet File Format

### 7.1.1:Write csv file

In [0]:
data.write.format("csv")\
    .mode("overwrite")\
    .option("header","true")\
    .save("/Volumes/dbacademy/default/data/output_csv")

Insight: Saved the cleaned dataset in csv format.

### 7.1.2:Write parquet file

In [0]:
data.write.format("parquet")\
    .mode("overwrite")\
    .option("header","true")\
    .save("/Volumes/dbacademy/default/data/output_parquet")

Insight: Saved the cleaned dataset in parquet format.

### 7.2:Compare both files with respect to reading time

#### 7.2.1:Read csv file and calculate time 

In [0]:
t1=time.time()

df1=spark.read.format("csv")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load("/Volumes/dbacademy/default/data/output_csv")
df1.count()
t2=time.time()


#### 7.2.2:Read parquet file and calculate time 

In [0]:
t3=time.time()
df2=spark.read.format("parquet")\
    .option("header","true")\
        .option("inferSchema","true")\
            .load("/Volumes/dbacademy/default/data/output_parquet")
df2.count()
t4=time.time()

### 7.3:Reading time difference between both file 

In [0]:
print(f"Time required to read csv file is {t2-t1} seconds")
print(f"Time required to read parquet file is {t4-t3} seconds")

Time required to read csv file is 1.5295295715332031 seconds
Time required to read parquet file is 0.912891149520874 seconds


## Step8:Complete Pipeline

In [0]:
complete_pipeline = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(schm)  # Define 'schm' with the appropriate schema before this line
    .load("/Volumes/dbacademy/default/data/retail_orders.csv")

    # Transformation
    .withColumn("rating", col("rating").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("age", col("age").cast("int"))

    # Remove null values
    .dropna(subset=["customer_name"])

    # Handling null values
    .fillna({
        "age": age_mean,
        "rating": rating_mean,
        "unit_price": 0,
        "quantity": 0,
        "city": "unknown",
        "state": "unknown",
        "product": "unknown",
        "category": "unknown",
        "payment_mode": "unknown",
        "delivery_status": "unknown"
    })

    # Handle invalid values
    .withColumn(
        "unit_price",
        when(col("unit_price") < 0, -col("unit_price")).otherwise(col("unit_price"))
    )
    .withColumn(
        "quantity",
        when(col("quantity") < 0, 0).otherwise(col("quantity"))
    )
    .withColumn(
        "rating",
        when(col("rating") < 0, 0).otherwise(col("rating"))
    )

    # Remove duplicates
    .dropDuplicates()

    # Rename columns
    .withColumnRenamed("customer_name", "customer_fullname")

    # Create new columns
    .withColumn("Total_amount", col("quantity") * col("unit_price"))
    .withColumn(
        "is_delivered",
        when(col("delivery_status") == "Delivered", 1).otherwise(0)
    )

    # Filter the data
    .filter((col("is_delivered") == 1) & (col("Total_amount") > 0))

    # Select required columns
    .select(
       "*"
    )
)
display(complete_pipeline)

order_id,customer_fullname,gender,age,city,state,order_date,product,category,quantity,unit_price,payment_mode,delivery_status,rating,returned,Total_amount,is_delivered
195,Matthew Nielsen,Female,49,Mumbai,Rajasthan,2024-02-08,Headphones,Accessories,1,29999,Crypto,Delivered,5,false,29999,1
262,Debra Hudson,Unknown,22,Ahmedabad,Maharashtra,12/01/2024,Keyboard,Accessories,3,599,unknown,Delivered,4,true,1797,1
375,Elijah Rodriguez,M,30,Ahmedabad,Rajasthan,2024-04-04,Mouse,Accessories,3,1499,Card,Delivered,8,true,4497,1
389,Shirley Sparks,F,150,Mumbai,Rajasthan,06-04-2023,Headphones,Accessories,3,1499,Card,Delivered,0,true,4497,1
395,Cody Charles,Male,22,Jaipur,Rajasthan,08-24-2023,Tablet,Electronics,2,29999,Crypto,Delivered,0,true,59998,1
446,Claire Farrell,F,-5,Ahmedabad,Delhi,05-19-2023,Keyboard,Accessories,2,1499,unknown,Delivered,4,false,2998,1
453,Jason Roberts,Female,30,Mumbai,Rajasthan,2023-04-05,Laptop,Electronics,2,1499,unknown,Delivered,3,true,2998,1
502,Nancy Matthews,Female,-5,Bengaluru,Rajasthan,30/05/2023,Printer,Electronics,3,500,Crypto,Delivered,4,true,1500,1
592,David Taylor,Female,22,Ahmedabad,Rajasthan,10/02/2024,unknown,Electronics,1,599,Card,Delivered,0,false,599,1
625,Heather Johnson,M,150,Delhi,Gujarat,07/06/2023,Laptop,Electronics,3,599,Cash,Delivered,0,null,1797,1


Insight:Creating the complete pipeline including loading,transforming,cleaning the dataset.

In [0]:
# Write CSV File
complete_pipeline.write.format("csv")\
            .mode("overwrite")\
            .option("header","true")\
            .save("/Volumes/dbacademy/default/data/pipeline_output_csv")

Insight:Saved the complete pipeline data in csv format.

In [0]:
# Write Parquet File
complete_pipeline.write.format("parquet")\
    .mode("overwrite")\
    .option("header","true")\
    .save("/Volumes/dbacademy/default/data/pipeline_output_parquet")

Insight:Saved the complete pipeline data in parquet format.